# Clothing Size Recommendation Advice System


## 1. Introduction & Problem Statement

### Introduction
With the rapid growth of online shopping, selecting the correct clothing size has become a major challenge for consumers. Unlike physical stores, online platforms do not allow customers to try garments before purchasing, which often leads to incorrect size selection, dissatisfaction, and high return rates. These issues negatively impact user experience, increase operational costs for retailers, and contribute to environmental waste due to excessive shipping and returns.

Advances in machine learning provide an opportunity to address this problem by analyzing user attributes and historical feedback to generate personalized size recommendations. By learning patterns from previous customers’ body measurements, product characteristics, and fit feedback, an intelligent advice system can assist users in choosing the most appropriate clothing size with higher confidence and accuracy.

### Problem Statement
The goal of this project is to build a **supervised machine learning–based advice system** that recommends an appropriate clothing size based on user and product attributes. Given structured input features such as body measurements, garment type, brand, and customer fit feedback, the system aims to classify whether a recommended size will result in a **good fit**, **tight fit**, or **loose fit**.

This problem is formulated as a **multi-class classification task**, where the target variable represents the perceived fit of a clothing item. The system will serve as a decision-support tool that enhances the online shopping experience by reducing size-related uncertainty and minimizing product returns. In Phase 1, the focus is on understanding the problem domain, exploring the dataset, preprocessing the data, and developing supervised learning models to form a reliable foundation for future enhancements in Phase 2.



## 2. Dataset Selection & Justification

### Dataset Goal & Source
To support the development of the clothing size recommendation system, we selected the **Clothing Fit Dataset for Size Recommendation**, which is publicly available on **Kaggle**.

**Dataset URL:**  
https://www.kaggle.com/datasets/rmisra/clothing-fit-dataset-for-size-recommendation

The primary goal of this dataset is to capture real customer experiences with clothing fit in online shopping contexts. It contains user-provided information and feedback that can be leveraged to predict whether a selected clothing size fits well.

### Dataset Description
The dataset consists of several hundred thousand records and includes structured, tabular data suitable for supervised learning. Key attributes include:

- User characteristics (e.g., height, weight, age)
- Product-related features (e.g., clothing category, brand)
- Size information selected by the customer
- Customer feedback indicating perceived fit (e.g., fit, small, large)

The target variable represents the fit outcome, which aligns directly with the classification objective of this project.

### Justification for Dataset Selection
This dataset was selected for the following reasons:

**1. Relevance to the Problem Domain**  
The dataset directly addresses the challenge of clothing size recommendation by linking user attributes and product information to fit outcomes. This makes it highly suitable for building an advice system focused on fashion and lifestyle guidance.

**2. Sufficient Size and Feature Complexity**  
The dataset contains a large number of observations and more than the minimum required number of features (10–15), ensuring adequate complexity for meaningful exploratory data analysis and model training.

**3. Structured and Cleanable Format**  
The data is provided in a structured, tabular format, making it compatible with Python-based data analysis and machine learning libraries. While preprocessing is required (e.g., handling missing values and encoding categorical variables), the dataset is well-suited for systematic cleaning and feature engineering.

**4. Clear Classification Labels**  
The presence of multiple fit categories enables the formulation of a multi-class classification problem, which satisfies the project requirement of having at least two classes and allows for richer evaluation and comparison of supervised learning algorithms.

**5. Real-World Applicability**  
Because the data is derived from real customer feedback, the resulting model has strong practical relevance and can be extended in Phase 2 to incorporate clustering and Generative AI for personalized explanations and recommendations.



## 3. Data inspection and Exploratory Data Analysis (EDA)



In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv(
    r'modcloth_final_data.csv',
    low_memory=False
)
df.shape

### 3.1 Dataset Overview

#### 3.1.1 Data Dimensions
According to the `df.shape` output, the dataset consists of **82,790 observations** and **18 features**. This substantial volume of data provides a solid foundation for training a multi-class classification model.
#### 3.1.2 Target Variable
The primary target variable for this project is **`fit`**. It is a categorical variable that classifies the sizing outcome into three distinct classes:
* **fit:** (56,757 observations) - The garment size was perfect.
* **large:** (13,059 observations) - The garment was larger/looser than expected.
* **small:** (12,974 observations) - The garment was smaller/tighter than expected.
#### 3.1.3 Data Types
The features in the dataset are classified into the following technical types:
* **Numerical (int64 / float64):** Includes `item_id`, `waist`, `size`, `quality`, `hips`, `bra size`, `user_id`, and `shoe size`.
* **Categorical / Text (object / str):** Includes `cup size`, `category`, `bust`, `height`, `user_name`, `length`, `fit`, `shoe width`, `review_summary`, and `review_text`.




### 3.2 Feature Analysis

This section provides a detailed statistical look at the numerical features and their relationships, which is essential for understanding the data distribution before building the advice system.

#####  3.2.1 Statistical summaries 
Based on the `describe()` output for the primary numerical features:
* **Bra Size:** Has a mean of **35.97** with a standard deviation of **3.22**, indicating a relatively consistent measurement across the dataset.
* **Quality:** The average rating is **3.95/5**, showing that most customers are satisfied with the product quality.
* **Size:** Shows high variability with a standard deviation of **8.27** and a range from **0 to 38**, reflecting the wide variety of clothing sizes available.
* **Shoe Size:** Provides additional physical context with an average of **8.15**.

| Feature | Count | Mean | Std Dev (std) | Min | Max |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Bra Size** | 76,772 | 35.97 | 3.22 | 28.0 | 48.0 |
| **Quality** | 82,722 | 3.95 | 0.99 | 1.0 | 5.0 |
| **Shoe Size** | 27,915 | 8.15 | 1.34 | 5.0 | 38.0 |
| **Size** | 82,790 | 12.66 | 8.27 | 0.0 | 38.0 |



##### 3.2.2 Correlation Analysis
Using the `corr()` method, we analyzed the relationships between these features:
* **Size vs. Bra Size:** A positive correlation exists here, confirming that as body measurements increase, the garment size typically follows.
* **Quality Independence:** The `quality` rating shows very low correlation with physical sizes, suggesting that "fit" issues (too small/too large) are independent of the perceived quality of the item.
* **Feature Relationships:** These correlations help in identifying which features are most redundant or most predictive for the target variable `fit`.

In [ ]:
summary_cols = ['bra size', 'quality', 'shoe size', 'size']
available_summary = [col for col in summary_cols if col in df.columns]

summary = df[available_summary].describe()
print(summary)
correlation_matrix = df[available_summary].corr()
print(correlation_matrix)

#### 3.2.3 Feature Distributions Analysis

Based on the visualized Histograms and Box Plots for the primary features, we can observe the following specific characteristics of the ModCloth data:

* **Size & Bra Size Distribution:** * The **Histogram for `size`** shows a wide distribution from 0 to 38, with a peak around sizes 8-12. This indicates that our model will have plenty of data for standard sizes but may need careful handling for the extreme ends of the spectrum.
    * The **Box Plot for `bra size`** shows that most users fall within the 34-38 range. Any values near 48 or 28 are clearly visible as potential outliers or represent a smaller segment of the customer base.

* **Quality (Ratings) Skewness:** * The **Histogram for `quality`** is heavily "Right-Skewed," meaning most ratings are concentrated at 4 and 5. This tells us that customers generally provide positive feedback, and our recommendation system should account for this bias in satisfaction levels.

* **Outlier Detection in Shoe Size:** * The **Box Plot for `shoe size`** reveals a significant finding: while the average is around 8.15, there are extreme values (up to 38). These are likely data entry errors (e.g., European sizes mixed with US sizes), which the Box Plot successfully highlighted for us to clean in the next phase.

* **Data Spread (Variance):** * The standard deviation of **8.27** in clothing `size` is visually evident in the spread of its Histogram, confirming a highly diverse range of product dimensions that our classification model must learn to navigate.

In [ ]:


plot_cols = ['bra size', 'quality', 'shoe size', 'size']

fig, axes = plt.subplots(nrows=len(plot_cols), ncols=2, figsize=(14, 18))

for i, col in enumerate(plot_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i, 0], color='teal')
    axes[i, 0].set_title(f'Distribution (Histogram) of {col}')
    
    sns.boxplot(x=df[col].dropna(), ax=axes[i, 1], color='orange')
    axes[i, 1].set_title(f'Visualizing Outliers (Box Plot) for {col}')

plt.tight_layout()
plt.show()

#### 3.2.4 Analysis of Feature-Target Relationships

This section explores how all primary features (Numerical and Categorical) correlate with the target variable **`fit`**. This comprehensive comparison identifies the most influential factors for the recommendation engine.

**1. Physical and Sizing Measurements (Numerical):**
* **Clothing Size & Bra Size:** The **Box Plots** indicate that "fit" satisfaction is generally consistent across most sizes. However, we observe more outliers and a wider spread in the **`small`** and **`large`** classes for users with higher measurements. This suggests that as the size increases, the standard deviation of actual garment dimensions grows, leading to more frequent fit issues.
* **Shoe Size:** Interestingly, shoe size shows a very stable relationship with fit across categories, suggesting it is a more standardized metric compared to upper-body clothing.

**2. Product and Body Type (Categorical):**
* **Category vs. Fit:** Analyzing the top 5 product categories (e.g., Dresses, Tops) reveals that **Dresses** tend to have a higher frequency of "small" and "large" reports compared to simpler items like "Tops." This indicates that complex garments with more measurement points are harder to fit accurately.
* **Cup Size:** Similar to bra size, the distribution of fit satisfaction across different **cup sizes** shows that users with larger cup sizes report "small" fit more frequently, highlighting a potential area for specific recommendation tuning.

**3. Feedback and Satisfaction (Quality):**
* **Quality vs. Fit:** There is a direct and strong relationship between fit and perceived quality. The **`fit`** class consistently shows higher median quality ratings (~4.0), while the **`small`** and **`large`** classes show a decrease in ratings. This confirms that fit accuracy is the primary driver of customer satisfaction in the ModCloth dataset.


In [ ]:


num_features = ['size', 'bra size', 'quality']
cat_features = ['category', 'cup size']

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20, 12))
plt.subplots_adjust(hspace=0.4)

for i, col in enumerate(num_features):
    sns.boxplot(x='fit', y=col, data=df, ax=axes[0, i], palette='Set2')
    axes[0, i].set_title(f'{col.capitalize()} vs Fit')


top_categories = df['category'].value_counts().nlargest(5).index
filtered_df = df[df['category'].isin(top_categories)]

sns.countplot(x='category', hue='fit', data=filtered_df, ax=axes[1, 0], palette='viridis')
axes[1, 0].set_title('Top Categories vs Fit')
axes[1, 0].tick_params(axis='x', rotation=45)

sns.countplot(x='cup size', hue='fit', data=df, ax=axes[1, 1], palette='viridis', 
              order=df['cup size'].value_counts().iloc[:7].index)
axes[1, 1].set_title('Top Cup Sizes vs Fit')

axes[1, 2].axis('off')

plt.show()

### 3.3 Visualization


#### 3.4 Target Variable Distribution

The majority of observations are labeled as "fit" (approximately 73.8%), while "small" and "large" each represent about 13–14% of the dataset.

This indicates moderate class imbalance. Although the imbalance is not extreme, evaluation metrics such as precision, recall, and F1-score will be important when assessing model performance.


In [ ]:


plt.figure(figsize=(6,4))
sns.countplot(x='fit', data=df)
plt.title("Distribution of Fit Categories")
plt.xlabel("Fit Category")
plt.ylabel("Count")
plt.show()


### 3.5 Missing Value Analysis

Several body-related attributes contain missing values, particularly:

- weight
- bust size
- body type
- age
- height

Since these features are likely important for predicting clothing fit, appropriate preprocessing strategies such as imputation or filtering will be necessary.

The target variable does not contain missing values, which is beneficial for supervised learning.


In [ ]:
plt.figure(figsize=(10,5))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values Heatmap")
plt.show()


### 3.6 Relationship Between Clothing Size and Fit

The boxplot shows noticeable variation in clothing size across fit categories. This suggests that the selected size plays an important role in determining whether an item fits properly.

This feature is likely to be one of the strongest predictors in the final model.


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x='fit', y='size', data=df)
plt.title("Clothing Size Across Fit Categories")
plt.show()


### 3.7 Relationship Between Bra Size and Fit

The distribution of bra sizes appears relatively consistent across the three fit categories (Small, Fit, Large). This suggests that while physical measurements like bra size are essential indicators, a single measurement alone may not strongly determine the final clothing fit. Instead, the fit is likely influenced by a combination of multiple body features (such as height, waist, and hips) along with the specific garment's dimensions.

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(x='fit', y='bra size', data=df)

plt.title("Bra Size Distribution Across Fit Categories")
plt.xlabel("Fit Category")
plt.ylabel("Bra Size")
plt.show()

### 3.8 Relationship Between Quality and Fit

The relationship between product quality and fit provides insights into customer satisfaction. Higher quality scores are more frequently associated with items labeled as "fit," whereas items categorized as "small" or "large" often receive lower quality ratings. This suggests that the perceived quality of a garment is significantly influenced by how well it fits the customer's body.

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(x='fit', y='quality', data=df)

plt.title("Quality Distribution Across Fit Categories")
plt.xlabel("Fit Category")
plt.ylabel("Quality Score")
plt.show()

### 3.9 Correlation Analysis

The correlation matrix provides a statistical overview of how numerical features in the ModCloth dataset relate to one another:

- **Size and Physical Measurements:** There is a moderate positive correlation between `size` and `bra size`. This is expected as larger body measurements generally necessitate larger clothing sizes, which validates the data's consistency.
- **Quality and Other Features:** The `quality` variable (customer rating) shows very weak correlation with physical measurements, suggesting that a customer's satisfaction with an item's quality is independent of their body size.
- **Multicollinearity:** No strong multicollinearity (correlation > 0.8) is observed between the independent variables at this stage. This is beneficial for our future classification models as it reduces redundancy.
- **Feature Engineering Potential:** While individual correlations are moderate, the interaction between features like `height`, `cup size`, and `waist` may reveal stronger predictive power for the "fit" outcome during the preprocessing phase.

In [ ]:
plt.figure(figsize=(8,6))


corr_features = ['bra size', 'quality', 'size']

sns.heatmap(df[corr_features].corr(), annot=True, cmap='coolwarm', fmt=".2f")

plt.title("Correlation Matrix of Numerical Features (ModCloth)")
plt.show()

### 3.10 Key Insights from EDA

From the exploratory analysis, several important conclusions can be drawn:

- The dataset is large and suitable for supervised machine learning.
- The target variable is moderately imbalanced but manageable.
- Clothing size shows a clear relationship with fit outcome.
- Body-related attributes are likely important predictors but contain missing values.
- Some numerical features contain extreme values and may require cleaning.
- Category and product type may also influence fit perception.

Overall, this analysis helped identify which features are most relevant for predicting clothing fit and provided direction for the preprocessing and modeling stages of the project.
